In [ ]:
import numpy as np

In [ ]:
import pandas as pd

In [ ]:
df=pd.read_csv('carss.csv')

In [ ]:
df.head()


# checking the data 

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df.describe

# now applying data preprocessing

In [ ]:
# replacing the  missing values with the mean of the columns only in the column km_driven
# cleaned_kms=df['kms_driven'].astype(str).str.replace('kms','',regex=False).str.replace(',','',regex=False)
# cleaned_kms=pd.to_numeric(cleaned_kms,errors='coerce')

# avg_kms_driven=cleaned_kms.mean()
# print('Average km driven:',avg_kms_driven)

# df['kms_driven']=cleaned_kms.fillna(avg_kms_driven)


In [ ]:
df.isnull().sum()

In [ ]:
# using sk learn 
cleaned_kms=df['kms_driven'].astype(str).str.replace('kms','',regex=False).str.replace(',','',regex=False)
df['kms_driven']=pd.to_numeric(cleaned_kms,errors='coerce')
from sklearn.impute import SimpleImputer
si=SimpleImputer(missing_values=np.nan,strategy='mean')
df[['kms_driven']]=si.fit_transform(df[['kms_driven']])

In [ ]:
df.isnull().sum()

In [ ]:

df.dtypes

In [ ]:
# now for the fuel type
si=SimpleImputer(missing_values=np.nan,strategy='most_frequent')
df[['fuel_type']]=si.fit_transform(df[['fuel_type']])


In [ ]:
df.isnull().sum()

In [ ]:
# now cleaning the price column
# remove commas and convert price to numeric
df['Price']=df['Price'].astype(str).str.replace(',','',regex=False)
df['Price']=pd.to_numeric(df['Price'],errors='coerce')
# setp 2 now drop the row  where Price in Nan
df.dropna(subset=['Price'],inplace=True,axis=0)
df.reset_index(drop=True,inplace=True)

# step 3 
print('null values in price:',df['Price'].isnull().sum())
print('new shape of data frame:',df.shape)

In [ ]:
# now for year
df['year']=pd.to_numeric(df['year'],errors='coerce')
df.dropna(subset=['year'],inplace=True)
df['year']=df['year'].astype(int)


In [ ]:
df.dtypes

In [ ]:
# reordering the columns
new_order=['name','company','year','kms_driven','fuel_type','Price']
df=df[new_order]
df

# now apply feature engineering



In [ ]:
# in this cell we only perfor slicing and joining 
# first we split the names and then we join it 
df['name']=df['name'].astype(str).str.split().str.slice(0,3).str.join(' ')
df['name'].head(10)

In [ ]:
#handling the categorical columns like fuel type
# now we can see that in our fuel type we have 3 types 
df['fuel_type'].value_counts()
df['company'].value_counts()

In [ ]:
# droping the name column because of too many unique car models
df=df.drop(columns=['name'])
df

In [ ]:
# training  and testing data seperation
X=df.iloc[:,:4]
X

In [ ]:
Y=df.iloc[:,-1]
Y

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42)

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
categorical_feature=['company','fuel_type']
transformer=ColumnTransformer([
    ('cat',OneHotEncoder(drop='first',sparse_output=False,handle_unknown='ignore'),categorical_feature)
],remainder='passthrough')
    

In [ ]:
X_train_transformed=transformer.fit_transform(X_train)
X_test_transformed=transformer.transform(X_test)

In [ ]:
X_train_transformed.shape


In [ ]:
X_test_transformed.shape

# train the model

In [168]:
# using ml algorithms for training the model 
#1 linearregression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score,mean_absolute_error
# lr model banaya
lr_model=LinearRegression()
# lr model pe transformed data ko fit kiya
lr_model.fit(X_train_transformed,Y_train)
# test data par prediction
Y_pred=lr_model.predict(X_test_transformed)
# perfromance
print('R2 score:',r2_score(Y_test,Y_pred))# R2 score yeh ek score hai jo batata hai ki model kitna sahi perform ya fit ho pa rha hai
print('Mean Absolute Error score:',mean_absolute_error(Y_test,Y_pred))# price prediction ke time avg kitne rupay ka error ya galti hai
 

R2 score: 0.7168148335173421
Mean Absolute Error score: 153916.89719602285


In [169]:
#2  ml algo
from sklearn.ensemble import RandomForestRegressor
rf_model=RandomForestRegressor(n_estimators=100,random_state=42)
rf_model.fit(X_train_transformed,Y_train)
Y_pred_rf=rf_model.predict(X_test_transformed)
print('R2 RandomForest Score:',r2_score(Y_test,Y_pred_rf))
# isko use nahi kiya kyu ki iska score bahut kam hai

R2 RandomForest Score: 0.35558810506448957


# testing the model


In [167]:
import pandas as pd

# sample liya hu kar ka
sample_car = pd.DataFrame({
    'company': ['Hyundai'],
    'year': [2017],
    'kms_driven': [45000.0],
    'fuel_type': ['Petrol']
})

# sample ko transform kiya hu
sample_transformed = transformer.transform(sample_car)

# Linear Regression model se kiya kyunki baaki yo se aacha accuracy/ score tha linr regression ka
predicted_price = lr_model.predict(sample_transformed)

print(f"Predicted Car Price: ₹ {predicted_price[0]:,.2f}")


Predicted Car Price: ₹ 445,547.48


# save the model using pickle


In [ ]:
import pickle

# Linear Regression model ko save kiya
with open('linear_regression_model.pkl', 'wb') as file:
    pickle.dump(lr_model, file)

# Transformer ko bhi save karna zaroori hai (taki naye data ko encode kar sakein)
with open('transformer.pkl', 'wb') as file:
    pickle.dump(transformer, file)

print("Model and Transformer successfully saved!")